In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
def fetch_forex_data(pair="EURUSD=X", period="2y", interval="1d"):
    """Downloads historical forex exchange rate data from Yahoo Finance."""
    print(f"[+] Fetching historical data for {pair}...")
    df = yf.download(tickers=pair, period=period, interval=interval, progress=False)
    
    # Clean MultiIndex columns if present in newer yfinance versions
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
        
    df.dropna(inplace=True)
    return df


In [4]:
def add_technical_indicators(df):
    """Calculates technical indicators to serve as AI feature inputs."""
    data = df.copy()
    
    # Exponential Moving Averages
    data['EMA_10'] = data['Close'].ewm(span=10, adjust=False).mean()
    data['EMA_50'] = data['Close'].ewm(span=50, adjust=False).mean()
    
    # Relative Strength Index (RSI)
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD & Signal Line
    ema_12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = ema_12 - ema_26
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    
    # Volatility / Returns
    data['Returns'] = data['Close'].pct_change()
    data['Volatility'] = data['Returns'].rolling(window=10).std()
    
    # Target Variable: 1 if next candle's Close is higher, 0 otherwise
    data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)
    
    data.dropna(inplace=True)
    return data

In [5]:
def train_forex_ai(df):
    """Trains a Random Forest classifier using chronological time-series splitting."""
    feature_cols = ['Open', 'High', 'Low', 'Close', 'EMA_10', 'EMA_50', 'RSI', 'MACD', 'MACD_Signal', 'Volatility']
    
    X = df[feature_cols]
    y = df['Target']
    
    # 80/20 Time-series split (Strict chronological order to prevent data leakage)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    
    print(f"[+] Training AI Model on {len(X_train)} historical candles...")
    
    model = RandomForestClassifier(
        n_estimators=150,
        max_depth=7,
        random_state=42,
        class_weight='balanced'
    )
    model.fit(X_train, y_train)
    
    # Evaluate performance
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    
    print("\n" + "="*45)
    print("      AI MODEL EVALUATION PERFORMANCE      ")
    print("="*45)
    print(f"Out-of-Sample Accuracy: {acc:.2%}\n")
    print(classification_report(y_test, predictions, target_names=['BEARISH (0)', 'BULLISH (1)']))
    print("="*45 + "\n")
    
    return model, feature_cols, X_test, y_test


In [6]:
def backtest_strategy(model, X_test, df_featured):
    """Simulates trading strategy returns based on AI predictions vs Buy & Hold."""
    test_df = df_featured.loc[X_test.index].copy()
    test_df['Signal'] = model.predict(X_test)
    
    # Convert binary prediction (1/0) into market exposure (1/-1)
    test_df['Position'] = np.where(test_df['Signal'] == 1, 1, -1)
    
    # Calculate strategy returns
    test_df['Strategy_Returns'] = test_df['Position'].shift(1) * test_df['Returns']
    
    cum_market = (1 + test_df['Returns']).cumprod() - 1
    cum_strategy = (1 + test_df['Strategy_Returns']).cumprod() - 1
    
    print("="*45)
    print("             BACKTEST RESULTS              ")
    print("="*45)
    print(f"Market Buy & Hold Return : {cum_market.iloc[-1]:.2%}")
    print(f"AI Strategy Return       : {cum_strategy.iloc[-1]:.2%}")
    print("="*45 + "\n")


In [8]:
def generate_latest_signal(model, feature_cols, df_featured, pair):
    """Outputs real-time directional prediction for the current active candle."""
    latest_row = df_featured[feature_cols].iloc[[-1]]
    latest_price = df_featured['Close'].iloc[-1]
    latest_rsi = df_featured['RSI'].iloc[-1]
    
    prediction = model.predict(latest_row)[0]
    probs = model.predict_proba(latest_row)[0]
    
    signal = "BUY (BULLISH)" if prediction == 1 else "SELL (BEARISH)"
    confidence = max(probs) * 100
    
    print("="*45)
    print(f"       LIVE FOREX SIGNAL: {pair}       ")
    print("="*45)
    print(f"Current Exchange Rate : {latest_price:.4f}")
    print(f"Current RSI (14)      : {latest_rsi:.2f}")
    print(f"AI Prediction         : {signal}")
    print(f"Confidence Level      : {confidence:.1f}%")
    print("="*45)


In [9]:
if __name__ == "__main__":
    PAIR = "EURUSD=X"  # Options: GBPUSD=X, USDJPY=X, AUDUSD=X, USDCAD=X
    PERIOD = "3y"       # Historical duration
    INTERVAL = "1d"     # Candle timeframe ('1d', '1h', '15m')
    
    # Step 1: Download Data
    raw_data = fetch_forex_data(pair=PAIR, period=PERIOD, interval=INTERVAL)
    
    # Step 2: Feature Engineering
    data_with_features = add_technical_indicators(raw_data)
    
    # Step 3: Train Model
    ai_model, features, X_test, y_test = train_forex_ai(data_with_features)
    
    # Step 4: Run Backtest
    backtest_strategy(ai_model, X_test, data_with_features)
    
    # Step 5: Output Real-Time Signal
    generate_latest_signal(ai_model, features, data_with_features, PAIR)

[+] Fetching historical data for EURUSD=X...
[+] Training AI Model on 613 historical candles...

      AI MODEL EVALUATION PERFORMANCE      
Out-of-Sample Accuracy: 56.49%

              precision    recall  f1-score   support

 BEARISH (0)       0.61      0.60      0.60        85
 BULLISH (1)       0.51      0.52      0.52        69

    accuracy                           0.56       154
   macro avg       0.56      0.56      0.56       154
weighted avg       0.57      0.56      0.57       154


             BACKTEST RESULTS              
Market Buy & Hold Return : -0.44%
AI Strategy Return       : 1.46%

       LIVE FOREX SIGNAL: EURUSD=X       
Current Exchange Rate : 1.1573
Current RSI (14)      : 71.15
AI Prediction         : SELL (BEARISH)
Confidence Level      : 54.5%
